# 05B — Feature Sets, Incident-Held-Out Splits, and Weighting Design

**Build ID:** `05B_V1_FEATURE_SETS_INCIDENT_SPLIT_WEIGHTING_DESIGN`

05A established a clean 304,143-unit conditional-hold modeling table.

05B freezes the experimental design **before any model is fitted**:

- scientifically interpretable predictor sets;
- 5 outer incident-held-out folds;
- 4 inner incident-held-out folds inside each outer-training set;
- ordinary, incident-equal, and incident-class-equal training weights;
- the evaluation protocol for unseen-fire discrimination and calibration.

**No predictive model is trained here. Random row splitting is prohibited.**

## Design principles

- `unit_uid` is never a predictor.
- `_irwin_norm` is the incident split key and never a predictor.
- `hold_label` is the target.
- `_engagement_norm` is outcome-derived and never a predictor.
- QA/runtime/provenance fields remain excluded.
- HRRR physical forecast values may be predictors; forecast availability,
  selected cycle, lead metadata, and fallback metadata are not.
- Predictors with more than 50% missingness are excluded from the primary
  feature sets but remain available in the master table.
- A ≤5%-missingness full set is retained as a coverage-stable robustness
  analysis.

## 0. Configuration

In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = (
    "05B_V1_FEATURE_SETS_INCIDENT_SPLIT_WEIGHTING_DESIGN"
)

from pathlib import Path
import sys
import subprocess
import importlib.util

PROJECT_ROOT = Path(
    os.environ.get("FIRELINE_PROJECT_ROOT", ".")
)

MASTER_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "modeling_table_assembly_v1" /
    "fireline_hold_modeling_master_v1.parquet"
)

COLUMN_MANIFEST_PATH = (
    PROJECT_ROOT / "outputs" /
    "modeling_table_assembly_v1" /
    "MODELING_COLUMN_MANIFEST_V1.csv"
)

MISSINGNESS_PATH = (
    PROJECT_ROOT / "outputs" /
    "modeling_table_assembly_v1" /
    "CANDIDATE_PREDICTOR_MISSINGNESS_V1.csv"
)

OUTPUT_ROOT = (
    PROJECT_ROOT / "outputs" /
    "modeling_design_v1"
)

PROCESSED_ROOT = (
    PROJECT_ROOT / "data" / "processed" /
    "modeling_design_v1"
)

for p in [OUTPUT_ROOT, PROCESSED_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

for p in [MASTER_PATH, COLUMN_MANIFEST_PATH, MISSINGNESS_PATH]:
    assert p.exists(), p

EXPECTED_UNITS = 304_143
EXPECTED_INCIDENTS = 381

OUTER_FOLDS = 5
INNER_FOLDS = 4
SPLIT_SEED = 20260825

PRIMARY_MAX_MISSING_PCT = 50.0
STABLE_MAX_MISSING_PCT = 5.0

print("=" * 100)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 100)
print("Master:", MASTER_PATH)

## 1. Imports

In [ ]:
required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "pyarrow": "pyarrow",
    "sklearn": "scikit-learn",
}

for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", package]
        )

from datetime import datetime, timezone
import json
import re

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyarrow.types as pat

from sklearn.model_selection import StratifiedGroupKFold

pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 500)
pd.set_option("display.width", 420)

## 2. Load 05A manifests and audit candidate predictor types

In [ ]:
column_manifest = pd.read_csv(COLUMN_MANIFEST_PATH)
missingness = pd.read_csv(MISSINGNESS_PATH)

master_meta = pq.read_metadata(MASTER_PATH)
master_schema = pq.read_schema(MASTER_PATH)

assert int(master_meta.num_rows) == EXPECTED_UNITS

schema_fields = {
    field.name: field
    for field in master_schema
}

candidate = (
    column_manifest[
        column_manifest["candidate_predictor"].astype(bool)
    ]
    .copy()
    .merge(
        missingness[
            [
                "column",
                "missing_count",
                "missing_pct",
                "dtype",
            ]
        ],
        on="column",
        how="left",
        validate="one_to_one",
    )
)

assert candidate["missing_pct"].notna().all()

def numeric_or_bool(column):
    t = schema_fields[column].type
    return bool(
        pat.is_integer(t)
        or pat.is_floating(t)
        or pat.is_boolean(t)
    )

candidate["numeric_or_bool"] = (
    candidate["column"].map(numeric_or_bool)
)

candidate["primary_coverage_ok"] = (
    candidate["missing_pct"] <= PRIMARY_MAX_MISSING_PCT
)

candidate["stable_coverage"] = (
    candidate["missing_pct"] <= STABLE_MAX_MISSING_PCT
)

candidate.to_csv(
    OUTPUT_ROOT / "CANDIDATE_PREDICTOR_AUDIT_V1.csv",
    index=False,
)

display(
    candidate.groupby(
        ["source", "numeric_or_bool", "primary_coverage_ok"],
        as_index=False,
    ).agg(
        columns=("column", "size"),
        max_missing_pct=("missing_pct", "max"),
    )
)

display(
    candidate[
        (~candidate["numeric_or_bool"])
        | (~candidate["primary_coverage_ok"])
    ][
        [
            "column",
            "source",
            "dtype",
            "missing_pct",
            "numeric_or_bool",
            "primary_coverage_ok",
        ]
    ]
)

## 3. Freeze physical predictor pools

### HRRR

Only physical weather values are admitted:

- U/V wind components;
- Earth-relative wind components;
- wind speed and gust;
- temperature and RH;
- toward-approach and cross-approach wind;
- physical window summaries.

Lead-hour numbers, valid-time rounding errors, cycle/fallback metadata, and
availability flags remain excluded.

Near weather = `t0`, +1 h, +3 h, +6 h, and 0–6 h summaries.

Extended weather adds +12 h, +18 h, ETA-relative points, and 0–12 h,
0–18 h, and 0–ETA summaries.

These are forecasts that were operationally available at the prediction anchor,
not realized future weather.

In [ ]:
eligible = candidate[
    candidate["numeric_or_bool"]
    & candidate["primary_coverage_ok"]
].copy()

# ---------------- Approach ----------------
APPROACH_DENY_TOKENS = (
    "label",
    "outcome",
    "challenge_end",
    "contact_time",
    "post_",
    "after_",
    "future_",
    "prediction_t1",
)

approach_pool = list(
    eligible.loc[
        (eligible["source"] == "approach")
        & ~eligible["column"].str.lower().apply(
            lambda x: any(
                token in x
                for token in APPROACH_DENY_TOKENS
            )
        ),
        "column",
    ]
)

# ---------------- HRRR ----------------
HRRR_POINT_VARS = (
    "u10_grid_mps",
    "v10_grid_mps",
    "u10_earth_east_mps",
    "v10_earth_north_mps",
    "wind10_speed_mps",
    "gust_mps",
    "t2_c",
    "rh2_pct",
    "wind10_toward_approach_mps",
    "wind10_cross_approach_mps",
)

NEAR_LABELS = ("t0", "p1h", "p3h", "p6h")
EXT_LABELS = (
    "p12h",
    "p18h",
    "eta25",
    "eta50",
    "eta75",
    "eta100",
)

WINDOW_STATS = (
    "t2_c_mean",
    "t2_c_max",
    "rh2_pct_mean",
    "rh2_pct_min",
    "wind10_speed_mean_mps",
    "wind10_speed_max_mps",
    "gust_mean_mps",
    "gust_max_mps",
    "toward_mean_mps",
    "toward_max_mps",
    "cross_mean_mps",
    "cross_abs_max_mps",
)

hrrr_candidate_set = set(
    eligible.loc[
        eligible["source"] == "hrrr",
        "column",
    ]
)

hrrr_near = []
for var in HRRR_POINT_VARS:
    for label in NEAR_LABELS:
        name = f"hrrr_{var}_{label}"
        if name in hrrr_candidate_set:
            hrrr_near.append(name)

for stat in WINDOW_STATS:
    name = f"hrrr_window_0_6h_{stat}"
    if name in hrrr_candidate_set:
        hrrr_near.append(name)

hrrr_extended = []
for var in HRRR_POINT_VARS:
    for label in EXT_LABELS:
        name = f"hrrr_{var}_{label}"
        if name in hrrr_candidate_set:
            hrrr_extended.append(name)

for window in ("12h", "18h", "eta"):
    for stat in WINDOW_STATS:
        name = f"hrrr_window_0_{window}_{stat}"
        if name in hrrr_candidate_set:
            hrrr_extended.append(name)

hrrr_selected = set(hrrr_near + hrrr_extended)
hrrr_unselected = sorted(
    hrrr_candidate_set - hrrr_selected
)

# ---------------- Terrain ----------------
terrain_pool = list(
    eligible.loc[
        eligible["source"] == "terrain",
        "column",
    ]
)

# ---------------- LANDFIRE ----------------
lf = eligible[
    eligible["source"] == "landfire"
].copy()

lf["tier_norm"] = (
    lf["landfire_tier"]
    .astype("string")
    .str.lower()
)

landfire_core = list(
    lf.loc[
        lf["tier_norm"].str.contains("core", na=False),
        "column",
    ]
)

landfire_extended = list(
    lf.loc[
        lf["tier_norm"].str.contains("extended", na=False),
        "column",
    ]
)

pool_summary = pd.DataFrame(
    [
        ["approach", len(approach_pool)],
        ["hrrr_near", len(hrrr_near)],
        ["hrrr_extended", len(hrrr_extended)],
        ["terrain", len(terrain_pool)],
        ["landfire_core", len(landfire_core)],
        ["landfire_extended", len(landfire_extended)],
        ["hrrr_candidate_not_selected", len(hrrr_unselected)],
    ],
    columns=["pool", "columns"],
)

display(pool_summary)

display(
    pd.DataFrame(
        {"hrrr_candidate_not_selected": hrrr_unselected}
    )
)

pool_summary.to_csv(
    OUTPUT_ROOT / "PHYSICAL_FEATURE_POOL_SUMMARY_V1.csv",
    index=False,
)

pd.DataFrame(
    {"column": hrrr_unselected}
).to_csv(
    OUTPUT_ROOT /
    "HRRR_CANDIDATES_EXCLUDED_FROM_PHYSICAL_POOL_V1.csv",
    index=False,
)

## 4. Freeze comparative feature sets

- **A_APPROACH_ONLY** — pre-contact dynamic fire behavior.
- **B_STATIC_ENV_CORE** — terrain + core pre-event LANDFIRE.
- **C_NEAR_WEATHER_ONLY** — near-term forecast weather only.
- **D_FULL_CORE** — approach + near weather + terrain + core LANDFIRE.
- **E_FULL_EXTENDED** — D plus extended HRRR and extended LANDFIRE.
- **F_FULL_COVERAGE_STABLE** — E restricted to ≤5% missingness.

Set F is a robustness analysis against performance driven primarily by
structured missingness or data availability.

In [ ]:
def dedupe(columns):
    return list(dict.fromkeys(columns))

FEATURE_SETS = {
    "A_APPROACH_ONLY": dedupe(
        approach_pool
    ),
    "B_STATIC_ENV_CORE": dedupe(
        terrain_pool
        + landfire_core
    ),
    "C_NEAR_WEATHER_ONLY": dedupe(
        hrrr_near
    ),
    "D_FULL_CORE": dedupe(
        approach_pool
        + hrrr_near
        + terrain_pool
        + landfire_core
    ),
    "E_FULL_EXTENDED": dedupe(
        approach_pool
        + hrrr_near
        + hrrr_extended
        + terrain_pool
        + landfire_core
        + landfire_extended
    ),
}

missing_pct = dict(
    zip(
        candidate["column"],
        candidate["missing_pct"],
    )
)

FEATURE_SETS["F_FULL_COVERAGE_STABLE"] = [
    c
    for c in FEATURE_SETS["E_FULL_EXTENDED"]
    if missing_pct[c] <= STABLE_MAX_MISSING_PCT
]

rows = []

candidate_lookup = candidate.set_index("column")

for set_name, columns in FEATURE_SETS.items():
    for col in columns:
        rec = candidate_lookup.loc[col]

        rows.append(
            {
                "feature_set": set_name,
                "column": col,
                "source": rec["source"],
                "missing_pct": rec["missing_pct"],
                "dtype": rec["dtype"],
                "landfire_tier": rec.get(
                    "landfire_tier",
                    None,
                ),
            }
        )

feature_set_manifest = pd.DataFrame(rows)

feature_set_summary = (
    feature_set_manifest.groupby(
        "feature_set",
        as_index=False,
    )
    .agg(
        columns=("column", "size"),
        median_missing_pct=("missing_pct", "median"),
        p90_missing_pct=(
            "missing_pct",
            lambda x: x.quantile(.90),
        ),
        max_missing_pct=("missing_pct", "max"),
    )
)

feature_set_source_counts = (
    feature_set_manifest.groupby(
        ["feature_set", "source"],
        as_index=False,
    )
    .agg(columns=("column", "size"))
)

display(feature_set_summary)
display(feature_set_source_counts)

assert all(
    len(cols) > 0
    and len(cols) == len(set(cols))
    for cols in FEATURE_SETS.values()
)

feature_set_manifest.to_csv(
    OUTPUT_ROOT / "FEATURE_SET_MANIFEST_V1.csv",
    index=False,
)

feature_set_summary.to_csv(
    OUTPUT_ROOT / "FEATURE_SET_SUMMARY_V1.csv",
    index=False,
)

feature_set_source_counts.to_csv(
    OUTPUT_ROOT / "FEATURE_SET_SOURCE_COUNTS_V1.csv",
    index=False,
)

with (
    OUTPUT_ROOT / "FEATURE_SETS_V1.json"
).open("w", encoding="utf-8") as f:
    json.dump(FEATURE_SETS, f, indent=2)

## 5. Build incident-level target profile

In [ ]:
model_index = pd.read_parquet(
    MASTER_PATH,
    columns=[
        "unit_uid",
        "_irwin_norm",
        "IncidentName",
        "Year",
        "hold_label",
    ],
)

assert len(model_index) == EXPECTED_UNITS
assert model_index["unit_uid"].is_unique
assert model_index["_irwin_norm"].notna().all()
assert model_index["hold_label"].isin([0, 1]).all()

incident_stats = (
    model_index.groupby(
        "_irwin_norm",
        as_index=False,
    )
    .agg(
        IncidentName=("IncidentName", "first"),
        Year=("Year", "first"),
        year_nunique=("Year", "nunique"),
        units=("unit_uid", "size"),
        held=("hold_label", "sum"),
    )
)

assert len(incident_stats) == EXPECTED_INCIDENTS
assert (incident_stats["year_nunique"] == 1).all()

incident_stats["burned_over"] = (
    incident_stats["units"]
    - incident_stats["held"]
)

incident_stats["held_rate"] = (
    incident_stats["held"]
    / incident_stats["units"]
)

incident_stats["class_profile"] = np.select(
    [
        (incident_stats["held"] > 0)
        & (incident_stats["burned_over"] > 0),
        incident_stats["burned_over"] == 0,
        incident_stats["held"] == 0,
    ],
    [
        "mixed",
        "held_only",
        "burned_over_only",
    ],
    default="unexpected",
)

incident_profile_summary = (
    incident_stats["class_profile"]
    .value_counts(dropna=False)
    .rename_axis("class_profile")
    .reset_index(name="incidents")
)

display(incident_profile_summary)

display(
    incident_stats[
        ["units", "held_rate"]
    ].describe(
        percentiles=[
            .1, .25, .5, .75, .9, .95, .99
        ]
    )
)

incident_stats.to_csv(
    OUTPUT_ROOT / "INCIDENT_TARGET_PROFILE_V1.csv",
    index=False,
)

incident_profile_summary.to_csv(
    OUTPUT_ROOT /
    "INCIDENT_CLASS_PROFILE_SUMMARY_V1.csv",
    index=False,
)

## 6. Freeze 5 outer incident-held-out folds

`StratifiedGroupKFold` is applied at unit level with `_irwin_norm` as the group.
Therefore the stratification objective sees the unit-level class distribution,
but **an entire wildfire is always assigned as one indivisible group**.

Every unit receives one outer fold, and every incident appears in exactly one
outer test fold.

In [ ]:
sgkf = StratifiedGroupKFold(
    n_splits=OUTER_FOLDS,
    shuffle=True,
    random_state=SPLIT_SEED,
)

outer_fold_by_unit = np.full(
    len(model_index),
    -1,
    dtype=np.int16,
)

X_dummy = np.zeros(
    (
        len(model_index),
        1,
    ),
    dtype=np.uint8,
)

y = model_index["hold_label"].to_numpy()
groups = model_index["_irwin_norm"].to_numpy()

for fold, (_, test_idx) in enumerate(
    sgkf.split(
        X_dummy,
        y,
        groups,
    )
):
    outer_fold_by_unit[
        test_idx
    ] = fold

assert (outer_fold_by_unit >= 0).all()

unit_outer = model_index.copy()
unit_outer["outer_fold"] = outer_fold_by_unit

incident_fold_nunique = (
    unit_outer.groupby(
        "_irwin_norm"
    )["outer_fold"]
    .nunique()
)

assert (incident_fold_nunique == 1).all()

outer_incident_folds = (
    unit_outer.groupby(
        "_irwin_norm",
        as_index=False,
    )
    .agg(
        IncidentName=("IncidentName", "first"),
        Year=("Year", "first"),
        outer_fold=("outer_fold", "first"),
        units=("unit_uid", "size"),
        held=("hold_label", "sum"),
    )
)

outer_incident_folds["burned_over"] = (
    outer_incident_folds["units"]
    - outer_incident_folds["held"]
)

outer_incident_folds["held_rate"] = (
    outer_incident_folds["held"]
    / outer_incident_folds["units"]
)

assert len(outer_incident_folds) == EXPECTED_INCIDENTS
assert outer_incident_folds["_irwin_norm"].is_unique
assert set(outer_incident_folds["outer_fold"]) == set(
    range(OUTER_FOLDS)
)

outer_incident_folds.to_csv(
    OUTPUT_ROOT / "OUTER_INCIDENT_FOLDS_V1.csv",
    index=False,
)

## 7. Outer-fold balance audit

In [ ]:
outer_fold_summary = (
    unit_outer.groupby(
        "outer_fold",
        as_index=False,
    )
    .agg(
        units=("unit_uid", "size"),
        incidents=("_irwin_norm", "nunique"),
        held=("hold_label", "sum"),
    )
)

outer_fold_summary["burned_over"] = (
    outer_fold_summary["units"]
    - outer_fold_summary["held"]
)

outer_fold_summary["held_rate"] = (
    outer_fold_summary["held"]
    / outer_fold_summary["units"]
)

target_units = EXPECTED_UNITS / OUTER_FOLDS

outer_fold_summary["unit_deviation_pct"] = (
    (
        outer_fold_summary["units"]
        - target_units
    )
    / target_units
    * 100.0
)

outer_year_units = (
    unit_outer.pivot_table(
        index="outer_fold",
        columns="Year",
        values="unit_uid",
        aggfunc="size",
        fill_value=0,
    )
    .reset_index()
)

display(outer_fold_summary)
display(outer_year_units)

assert int(
    outer_fold_summary["units"].sum()
) == EXPECTED_UNITS

assert int(
    outer_fold_summary["incidents"].sum()
) == EXPECTED_INCIDENTS

assert outer_fold_summary["held"].gt(0).all()
assert outer_fold_summary["burned_over"].gt(0).all()

year_cols = [
    c
    for c in outer_year_units.columns
    if c != "outer_fold"
]

assert (outer_year_units[year_cols] > 0).all().all()

outer_fold_summary.to_csv(
    OUTPUT_ROOT / "OUTER_FOLD_BALANCE_V1.csv",
    index=False,
)

outer_year_units.to_csv(
    OUTPUT_ROOT / "OUTER_FOLD_YEAR_BALANCE_V1.csv",
    index=False,
)

## 8. Freeze 4 inner incident-held-out folds within each outer training set

For each outer fold, its test incidents are removed first. The remaining
incidents are then repartitioned into 4 grouped inner folds.

Any preprocessing, hyperparameter selection, feature tuning, or probability
threshold selection must occur only inside these inner folds.

In [ ]:
inner_rows = []

for outer_fold in range(OUTER_FOLDS):

    train_mask = (
        unit_outer["outer_fold"]
        != outer_fold
    )

    train = unit_outer.loc[
        train_mask
    ].copy()

    inner_sgkf = StratifiedGroupKFold(
        n_splits=INNER_FOLDS,
        shuffle=True,
        random_state=(
            SPLIT_SEED
            + 1000
            + outer_fold
        ),
    )

    inner_fold = np.full(
        len(train),
        -1,
        dtype=np.int16,
    )

    train = train.reset_index(drop=True)

    X_inner = np.zeros(
        (
            len(train),
            1,
        ),
        dtype=np.uint8,
    )

    for f, (_, val_idx) in enumerate(
        inner_sgkf.split(
            X_inner,
            train["hold_label"].to_numpy(),
            train["_irwin_norm"].to_numpy(),
        )
    ):
        inner_fold[val_idx] = f

    assert (inner_fold >= 0).all()

    train["inner_fold"] = inner_fold

    incident_check = (
        train.groupby(
            "_irwin_norm"
        )["inner_fold"]
        .nunique()
    )

    assert (incident_check == 1).all()

    incident_map = (
        train.groupby(
            "_irwin_norm",
            as_index=False,
        )
        .agg(
            Year=("Year", "first"),
            inner_fold=("inner_fold", "first"),
            units=("unit_uid", "size"),
            held=("hold_label", "sum"),
        )
    )

    incident_map["burned_over"] = (
        incident_map["units"]
        - incident_map["held"]
    )

    incident_map["outer_fold"] = outer_fold

    inner_rows.append(incident_map)

inner_incident_folds = pd.concat(
    inner_rows,
    ignore_index=True,
)

expected_inner_rows = (
    sum(
        EXPECTED_INCIDENTS
        - int(
            (
                outer_incident_folds["outer_fold"]
                == outer_fold
            ).sum()
        )
        for outer_fold in range(OUTER_FOLDS)
    )
)

assert len(inner_incident_folds) == expected_inner_rows

for outer_fold in range(OUTER_FOLDS):

    outer_test_incidents = set(
        outer_incident_folds.loc[
            outer_incident_folds["outer_fold"]
            == outer_fold,
            "_irwin_norm",
        ]
    )

    inner_here = inner_incident_folds[
        inner_incident_folds["outer_fold"]
        == outer_fold
    ]

    assert not (
        set(inner_here["_irwin_norm"])
        & outer_test_incidents
    )

    assert inner_here["_irwin_norm"].is_unique

    assert set(inner_here["inner_fold"]) == set(
        range(INNER_FOLDS)
    )

inner_balance = (
    inner_incident_folds.groupby(
        ["outer_fold", "inner_fold"],
        as_index=False,
    )
    .agg(
        incidents=("_irwin_norm", "size"),
        units=("units", "sum"),
        held=("held", "sum"),
        burned_over=("burned_over", "sum"),
    )
)

inner_balance["held_rate"] = (
    inner_balance["held"]
    / inner_balance["units"]
)

display(inner_balance)

assert inner_balance["held"].gt(0).all()
assert inner_balance["burned_over"].gt(0).all()

inner_incident_folds.to_csv(
    OUTPUT_ROOT /
    "INNER_INCIDENT_FOLDS_LONG_V1.csv",
    index=False,
)

inner_balance.to_csv(
    OUTPUT_ROOT / "INNER_FOLD_BALANCE_V1.csv",
    index=False,
)

## 9. Freeze three training-weight regimes

### Ordinary row weighting
`w = 1`

### Incident-equal weighting
Each incident contributes total raw weight 1:
`w = 1 / n_incident`

### Incident-class-equal weighting
For a mixed incident, Held and Burned Over each contribute total raw weight
0.5. For a single-class incident, the observed class contributes total raw
weight 1.

The stored weights are relative/raw. A later training notebook may multiply
all training weights by one common fold-specific constant without changing
their meaning.

In [ ]:
unit_design = unit_outer.merge(
    incident_stats[
        [
            "_irwin_norm",
            "units",
            "held",
            "burned_over",
            "class_profile",
        ]
    ],
    on="_irwin_norm",
    how="left",
    validate="many_to_one",
)

unit_design["weight_row_raw"] = 1.0

unit_design["weight_incident_equal_raw"] = (
    1.0
    / unit_design["units"].astype(float)
)

held_count = unit_design["held"].astype(float)
bo_count = unit_design["burned_over"].astype(float)

mixed = (
    (held_count > 0)
    & (bo_count > 0)
)

unit_design[
    "weight_incident_class_equal_raw"
] = np.where(
    mixed,
    np.where(
        unit_design["hold_label"] == 1,
        0.5 / held_count,
        0.5 / bo_count,
    ),
    np.where(
        unit_design["hold_label"] == 1,
        1.0 / held_count,
        1.0 / bo_count,
    ),
)

weight_cols = [
    "weight_row_raw",
    "weight_incident_equal_raw",
    "weight_incident_class_equal_raw",
]

assert np.isfinite(
    unit_design[weight_cols].to_numpy(float)
).all()

assert (unit_design[weight_cols] > 0).all().all()

incident_equal_totals = (
    unit_design.groupby("_irwin_norm")[
        "weight_incident_equal_raw"
    ].sum()
)

assert np.allclose(
    incident_equal_totals.to_numpy(float),
    1.0,
    atol=1e-12,
    rtol=0,
)

mixed_incidents = set(
    incident_stats.loc[
        incident_stats["class_profile"]
        == "mixed",
        "_irwin_norm",
    ]
)

mixed_class_totals = (
    unit_design[
        unit_design["_irwin_norm"].isin(
            mixed_incidents
        )
    ]
    .groupby(
        ["_irwin_norm", "hold_label"]
    )[
        "weight_incident_class_equal_raw"
    ]
    .sum()
)

assert np.allclose(
    mixed_class_totals.to_numpy(float),
    0.5,
    atol=1e-12,
    rtol=0,
)

UNIT_DESIGN_PATH = (
    PROCESSED_ROOT /
    "unit_outer_fold_and_training_weights_v1.parquet"
)

unit_design[
    [
        "unit_uid",
        "_irwin_norm",
        "IncidentName",
        "Year",
        "hold_label",
        "outer_fold",
        *weight_cols,
    ]
].to_parquet(
    UNIT_DESIGN_PATH,
    index=False,
)

weight_summary = pd.DataFrame(
    [
        {
            "scheme": "row",
            "total_raw_weight":
            unit_design["weight_row_raw"].sum(),
        },
        {
            "scheme": "incident_equal",
            "total_raw_weight":
            unit_design[
                "weight_incident_equal_raw"
            ].sum(),
        },
        {
            "scheme": "incident_class_equal",
            "total_raw_weight":
            unit_design[
                "weight_incident_class_equal_raw"
            ].sum(),
        },
    ]
)

display(weight_summary)

weight_summary.to_csv(
    OUTPUT_ROOT /
    "TRAINING_WEIGHT_SUMMARY_V1.csv",
    index=False,
)

print("Saved:", UNIT_DESIGN_PATH)

## 10. Freeze evaluation protocol

Every reported prediction must be an **outer-fold out-of-fold prediction** from
a model that did not see that wildfire during training.

### Probabilistic metrics
- pooled Brier score;
- pooled log loss;
- incident-macro Brier score;
- incident-macro log loss.

### Discrimination
- pooled PR-AUC;
- pooled ROC-AUC;
- macro PR-AUC across mixed-class incidents;
- macro ROC-AUC across mixed-class incidents.

Single-class incidents stay in probabilistic metrics. AUC is not fabricated for
them because it is mathematically undefined.

### Calibration
- calibration intercept;
- calibration slope;
- reliability bins / reliability plot;
- ECE as a descriptive supplement.

### Threshold metrics
If classification metrics are reported, the threshold must be selected using
inner CV only. The outer test fold must never choose a threshold.

In [ ]:
EVALUATION_PROTOCOL = {
    "split_unit": "incident/_irwin_norm",
    "outer_folds": OUTER_FOLDS,
    "inner_folds": INNER_FOLDS,
    "random_row_split_prohibited": True,
    "outer_test_rule": (
        "Each incident appears in exactly one outer test fold."
    ),
    "inner_selection_rule": (
        "Only outer-training incidents participate in inner CV."
    ),
    "weighting_schemes": [
        "row",
        "incident_equal",
        "incident_class_equal",
    ],
    "probabilistic_metrics": [
        "pooled_brier",
        "pooled_log_loss",
        "incident_macro_brier",
        "incident_macro_log_loss",
    ],
    "discrimination_metrics": [
        "pooled_pr_auc",
        "pooled_roc_auc",
        "mixed_incident_macro_pr_auc",
        "mixed_incident_macro_roc_auc",
    ],
    "calibration_metrics": [
        "calibration_intercept",
        "calibration_slope",
        "reliability_bins",
        "ece_descriptive",
    ],
    "threshold_rule": (
        "Choose threshold on inner CV only, never outer test."
    ),
    "primary_max_missing_pct": PRIMARY_MAX_MISSING_PCT,
    "coverage_stable_max_missing_pct":
    STABLE_MAX_MISSING_PCT,
    "explicit_availability_qa_predictors": False,
    "primary_claim": (
        "Generalization to unseen wildfire incidents."
    ),
}

with (
    OUTPUT_ROOT / "EVALUATION_PROTOCOL_V1.json"
).open("w", encoding="utf-8") as f:
    json.dump(
        EVALUATION_PROTOCOL,
        f,
        indent=2,
    )

print(
    json.dumps(
        EVALUATION_PROTOCOL,
        indent=2,
    )
)

## 11. Final design checks

In [ ]:
forbidden = {
    "hold_label",
    "_engagement_norm",
    "_irwin_norm",
    "unit_uid",
    "IncidentName",
}

checks = [
    {
        "check": "exact_304143_units",
        "pass": len(model_index) == EXPECTED_UNITS,
    },
    {
        "check": "exact_381_incidents",
        "pass": len(incident_stats) == EXPECTED_INCIDENTS,
    },
    {
        "check": "outer_incidents_grouped_once",
        "pass": bool(
            (incident_fold_nunique == 1).all()
        ),
    },
    {
        "check": "all_five_outer_folds_present",
        "pass": set(
            outer_incident_folds["outer_fold"]
        ) == set(range(OUTER_FOLDS)),
    },
    {
        "check": "outer_units_sum_304143",
        "pass": int(
            outer_fold_summary["units"].sum()
        ) == EXPECTED_UNITS,
    },
    {
        "check": "outer_incidents_sum_381",
        "pass": int(
            outer_fold_summary["incidents"].sum()
        ) == EXPECTED_INCIDENTS,
    },
    {
        "check": "every_outer_fold_has_both_classes",
        "pass": bool(
            outer_fold_summary["held"].gt(0).all()
            and outer_fold_summary[
                "burned_over"
            ].gt(0).all()
        ),
    },
    {
        "check": "every_outer_fold_has_all_years",
        "pass": bool(
            (
                outer_year_units[year_cols] > 0
            ).all().all()
        ),
    },
    {
        "check": "inner_splits_exclude_outer_test_incidents",
        "pass": True,
    },
    {
        "check": "incident_equal_weight_total_one",
        "pass": bool(
            np.allclose(
                incident_equal_totals.to_numpy(float),
                1.0,
                atol=1e-12,
                rtol=0,
            )
        ),
    },
    {
        "check": "mixed_incident_class_total_half",
        "pass": bool(
            np.allclose(
                mixed_class_totals.to_numpy(float),
                0.5,
                atol=1e-12,
                rtol=0,
            )
        ),
    },
    {
        "check": "all_feature_sets_nonempty",
        "pass": all(
            len(v) > 0
            for v in FEATURE_SETS.values()
        ),
    },
    {
        "check": "primary_full_set_at_most_50pct_missing",
        "pass": all(
            missing_pct[c]
            <= PRIMARY_MAX_MISSING_PCT
            for c in FEATURE_SETS[
                "E_FULL_EXTENDED"
            ]
        ),
    },
    {
        "check": "stable_set_at_most_5pct_missing",
        "pass": all(
            missing_pct[c]
            <= STABLE_MAX_MISSING_PCT
            for c in FEATURE_SETS[
                "F_FULL_COVERAGE_STABLE"
            ]
        ),
    },
    {
        "check": "forbidden_fields_absent_from_all_feature_sets",
        "pass": not any(
            c in forbidden
            for cols in FEATURE_SETS.values()
            for c in cols
        ),
    },
    {
        "check": "random_row_split_prohibited",
        "pass": bool(
            EVALUATION_PROTOCOL[
                "random_row_split_prohibited"
            ]
        ),
    },
]

checks_df = pd.DataFrame(checks)

display(checks_df)

DESIGN_PASS = bool(
    checks_df["pass"].all()
)

print()
print(
    "05B EXPERIMENTAL DESIGN:",
    (
        "PASS / READY FOR INCIDENT-HELD-OUT BASELINE MODELING"
        if DESIGN_PASS
        else "REVIEW REQUIRED"
    ),
)

checks_df.to_csv(
    OUTPUT_ROOT / "FINAL_DESIGN_CHECKS_V1.csv",
    index=False,
)

## 12. Final report

In [ ]:
def tomd(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_string(index=False)

report = f"""# 05B Feature Sets, Incident-Held-Out Splits, and Weighting Design

Build: `{NOTEBOOK_BUILD_ID}`

Generated UTC: {datetime.now(timezone.utc).isoformat()}

## Cohort

Units: **{len(model_index):,}**

Incidents: **{len(incident_stats):,}**

## Incident class profiles

{tomd(incident_profile_summary)}

## Physical predictor pools

{tomd(pool_summary)}

## Feature-set summary

{tomd(feature_set_summary)}

## Feature-set source composition

{tomd(feature_set_source_counts)}

## Outer-fold balance

{tomd(outer_fold_summary)}

## Outer-fold year balance

{tomd(outer_year_units)}

## Inner-fold balance

{tomd(inner_balance)}

## Training-weight summary

{tomd(weight_summary)}

## Final checks

{tomd(checks_df)}

## Decision

`{"PASS / READY FOR INCIDENT-HELD-OUT BASELINE MODELING" if DESIGN_PASS else "REVIEW REQUIRED"}`

No predictive model was fitted in 05B.
"""

REPORT_PATH = (
    OUTPUT_ROOT / "MODELING_DESIGN_REPORT_V1.md"
)

REPORT_PATH.write_text(
    report,
    encoding="utf-8",
)

print(report)
print()
print("Saved:", REPORT_PATH)

## 13. Output inventory

In [ ]:
print("=" * 100)
print("05B OUTPUTS")
print("=" * 100)

for p in sorted(OUTPUT_ROOT.glob("*")):
    if p.is_file():
        print(
            f"{p.name:<90}"
            f"{p.stat().st_size/(1024**2):>12.3f} MB"
        )

for p in sorted(PROCESSED_ROOT.glob("*")):
    if p.is_file():
        print(
            f"{p.name:<90}"
            f"{p.stat().st_size/(1024**2):>12.3f} MB"
        )

print()
print("DONE:", NOTEBOOK_BUILD_ID)